# UC02 Group Booking Assistant — Backtest Report

**Time-based holdout evaluation: LightGBM (calibrated) vs Logistic Regression vs Rule-based Baseline**

| Split | Months | Purpose |
|-------|--------|---------|
| Train | Jun–Sep | Model training |
| Validation | Oct | Isotonic calibration |
| **Holdout** | **Nov** | **Evaluation (never seen during training)** |


In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import shap
import joblib
from pathlib import Path

from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss,
    roc_curve, precision_recall_curve, confusion_matrix,
    classification_report
)
from sklearn.calibration import calibration_curve

from etl.features import build_features, get_X_y
from models.train_conversion import time_split
from engines.baseline import baseline_decision

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
ARTIFACTS = Path('../models/artifacts')
print('Libraries loaded ✓')

## 1. Load Data & Model Artifacts

In [ ]:
# Load features (from cache)
feat = build_features(use_cache=True)
train_df, val_df, holdout_df = time_split(feat)

X_train, y_train = get_X_y(train_df, exclude_live=False)
X_val,   y_val   = get_X_y(val_df,   exclude_live=False)
X_hold,  y_hold  = get_X_y(holdout_df, exclude_live=False)

print(f"Train   : {len(X_train):>5} rows | {y_train.sum():>3} converted ({100*y_train.mean():.1f}%)")
print(f"Val     : {len(X_val):>5} rows | {y_val.sum():>3} converted ({100*y_val.mean():.1f}%)")
print(f"Holdout : {len(X_hold):>5} rows | {y_hold.sum():>3} converted ({100*y_hold.mean():.1f}%)")

In [ ]:
# Load saved artifacts
model       = joblib.load(ARTIFACTS / 'lgbm_model.pkl')
calibrator  = joblib.load(ARTIFACTS / 'calibrator.pkl')
lr          = joblib.load(ARTIFACTS / 'lr_model.pkl')
explainer   = joblib.load(ARTIFACTS / 'shap_explainer.pkl')
feat_names  = joblib.load(ARTIFACTS / 'feature_names.pkl')

print('Artifacts loaded ✓')
print(f'Features used ({len(feat_names)}): {feat_names}')

## 2. Generate Predictions

In [ ]:
# LightGBM — raw + calibrated
lgbm_raw_hold  = model.predict_proba(X_hold.fillna(-1))[:, 1]
lgbm_cal_hold  = calibrator.transform(lgbm_raw_hold)

lgbm_raw_val   = model.predict_proba(X_val.fillna(-1))[:, 1]
lgbm_cal_val   = calibrator.transform(lgbm_raw_val)

# Logistic Regression
lr_hold = lr.predict_proba(X_hold.fillna(-1))[:, 1]
lr_val  = lr.predict_proba(X_val.fillna(-1))[:, 1]

# Baseline rule-based (naive 10% P(convert) if Accept/Partial, 0 if Reject)
def baseline_probs(df):
    probs = []
    for _, row in df.iterrows():
        result = baseline_decision(
            group_size=row.get('group_size', 20),
            load_factor=row.get('load_factor', float('nan')),
            ref_fare=row.get('ref_fare', float('nan')),
            capacity=row.get('total_capacity', float('nan')),
        )
        probs.append(result['p_convert_estimate'])
    return np.array(probs)

base_hold = baseline_probs(holdout_df.reset_index(drop=True))

print('Predictions generated ✓')

## 3. Metrics Summary

In [ ]:
def metrics(y_true, y_prob, name):
    roc  = roc_auc_score(y_true, y_prob)
    pr   = average_precision_score(y_true, y_prob)
    brier = brier_score_loss(y_true, y_prob)
    return {'Model': name, 'ROC-AUC': roc, 'PR-AUC': pr, 'Brier Score': brier}

rows = [
    metrics(y_hold, lgbm_cal_hold,  'LightGBM (calibrated) ★'),
    metrics(y_hold, lgbm_raw_hold,  'LightGBM (raw)'),
    metrics(y_hold, lr_hold,        'Logistic Regression'),
    metrics(y_hold, base_hold,      'Rule-based Baseline'),
]

summary = pd.DataFrame(rows).set_index('Model')
summary['ROC-AUC']    = summary['ROC-AUC'].map('{:.4f}'.format)
summary['PR-AUC']     = summary['PR-AUC'].map('{:.4f}'.format)
summary['Brier Score'] = summary['Brier Score'].map('{:.4f}'.format)

print('═' * 65)
print('HOLDOUT METRICS (November — never seen during training)')
print('═' * 65)
print(summary.to_string())
print('═' * 65)
print('Note: Brier Score — lower is better | ROC/PR-AUC — higher is better')
print(f'Base rate (Nov): {100*float(y_hold.mean()):.1f}% — random classifier ROC-AUC = 0.500')

## 4. ROC & Precision-Recall Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── ROC Curve ──
ax = axes[0]
for y_prob, label, color, ls in [
    (lgbm_cal_hold, 'LightGBM calibrated', '#1f77b4', '-'),
    (lgbm_raw_hold, 'LightGBM raw',        '#aec7e8', '--'),
    (lr_hold,       'Logistic Regression', '#ff7f0e', '-.'),
    (base_hold,     'Rule-based baseline', '#d62728', ':'),
]:
    fpr, tpr, _ = roc_curve(y_hold, y_prob)
    auc = roc_auc_score(y_hold, y_prob)
    ax.plot(fpr, tpr, color=color, linestyle=ls, lw=2, label=f'{label} (AUC={auc:.3f})')

ax.plot([0,1],[0,1],'k--', lw=1, alpha=0.5, label='Random (AUC=0.500)')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — November Holdout')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# ── Precision-Recall Curve ──
ax = axes[1]
base_rate = float(y_hold.mean())
for y_prob, label, color, ls in [
    (lgbm_cal_hold, 'LightGBM calibrated', '#1f77b4', '-'),
    (lgbm_raw_hold, 'LightGBM raw',        '#aec7e8', '--'),
    (lr_hold,       'Logistic Regression', '#ff7f0e', '-.'),
    (base_hold,     'Rule-based baseline', '#d62728', ':'),
]:
    p, r, _ = precision_recall_curve(y_hold, y_prob)
    ap = average_precision_score(y_hold, y_prob)
    ax.plot(r, p, color=color, linestyle=ls, lw=2, label=f'{label} (AP={ap:.3f})')

ax.axhline(base_rate, color='k', linestyle='--', lw=1, alpha=0.5,
           label=f'No-skill ({base_rate:.3f})')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve — November Holdout')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../eval/roc_pr_curves.png', bbox_inches='tight')
plt.show()
print('Saved → eval/roc_pr_curves.png')

## 5. Calibration Plot

A well-calibrated model should have predicted probability ≈ actual conversion rate.
Isotonic calibration corrects LightGBM's tendency to push probabilities toward extremes.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

n_bins = 8
for y_prob, label, color, ls in [
    (lgbm_cal_hold, 'LightGBM calibrated', '#1f77b4', '-o'),
    (lgbm_raw_hold, 'LightGBM raw',        '#aec7e8', '--s'),
    (lr_hold,       'Logistic Regression', '#ff7f0e', '-.^'),
]:
    prob_true, prob_pred = calibration_curve(y_hold, y_prob, n_bins=n_bins, strategy='quantile')
    ax.plot(prob_pred, prob_true, ls, color=color, lw=2, markersize=7, label=label)

ax.plot([0, 1], [0, 1], 'k--', lw=1.5, alpha=0.6, label='Perfect calibration')
ax.set_xlabel('Mean Predicted Probability')
ax.set_ylabel('Fraction of Positives (Actual Conversion Rate)')
ax.set_title('Reliability / Calibration Plot — November Holdout')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
ax.set_xlim(0, 0.5)
ax.set_ylim(0, 0.5)

plt.tight_layout()
plt.savefig('../eval/calibration_plot.png', bbox_inches='tight')
plt.show()
print('Saved → eval/calibration_plot.png')

## 6. SHAP — Global Feature Importance

In [ ]:
# Compute SHAP values on holdout
X_hold_filled = X_hold.fillna(-1)
shap_values = explainer.shap_values(X_hold_filled)

# shap_values may be list (for binary classification, take index 1)
if isinstance(shap_values, list):
    sv = shap_values[1]
else:
    sv = shap_values

mean_abs_shap = np.abs(sv).mean(axis=0)
importance_df = pd.DataFrame({
    'feature': feat_names,
    'mean_abs_shap': mean_abs_shap
}).sort_values('mean_abs_shap', ascending=False)

fig, ax = plt.subplots(figsize=(8, 6))
colors = ['#1f77b4' if i < 5 else '#aec7e8' for i in range(len(importance_df))]
ax.barh(importance_df['feature'][::-1], importance_df['mean_abs_shap'][::-1], color=colors[::-1])
ax.set_xlabel('Mean |SHAP Value| (impact on model output)')
ax.set_title('Global Feature Importance — SHAP (Holdout Set)')
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('../eval/shap_importance.png', bbox_inches='tight')
plt.show()

print('\nTop 5 features by SHAP:')
print(importance_df.head(5).to_string(index=False))
print('\nSaved → eval/shap_importance.png')

## 7. SHAP Beeswarm Plot

In [ ]:
shap_exp = shap.Explanation(
    values=sv,
    base_values=explainer.expected_value if not isinstance(explainer.expected_value, list)
                else explainer.expected_value[1],
    data=X_hold_filled.values,
    feature_names=feat_names
)

plt.figure(figsize=(10, 6))
shap.plots.beeswarm(shap_exp, max_display=15, show=False)
plt.title('SHAP Beeswarm — Feature Impact Direction (Holdout)')
plt.tight_layout()
plt.savefig('../eval/shap_beeswarm.png', bbox_inches='tight')
plt.show()
print('Saved → eval/shap_beeswarm.png')

## 8. Fare Ratio Analysis — Converted vs Not Converted

In [ ]:
hold_full = holdout_df.copy()
hold_full['p_convert'] = lgbm_cal_hold
hold_full['y'] = y_hold.values

fare_ratio_valid = hold_full.dropna(subset=['fare_ratio'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution of fare_ratio
ax = axes[0]
converted   = fare_ratio_valid[fare_ratio_valid['y'] == 1]['fare_ratio']
not_convert = fare_ratio_valid[fare_ratio_valid['y'] == 0]['fare_ratio']

ax.hist(not_convert.clip(0.5, 1.5), bins=20, alpha=0.6, color='#d62728', label=f'Not converted (n={len(not_convert)})', density=True)
ax.hist(converted.clip(0.5, 1.5),   bins=20, alpha=0.7, color='#2ca02c', label=f'Converted (n={len(converted)})', density=True)
ax.axvline(1.0, color='k', linestyle='--', lw=1.5, alpha=0.6, label='Fare = reference')
ax.set_xlabel('Fare Ratio (Quoted / Reference Fare)')
ax.set_ylabel('Density')
ax.set_title('Fare Ratio Distribution by Outcome (Holdout)')
ax.legend()
ax.grid(alpha=0.3)

if len(converted) > 0 and len(not_convert) > 0:
    print(f"Fare ratio — converted:     {converted.mean():.3f} (mean) | {converted.median():.3f} (median)")
    print(f"Fare ratio — not converted: {not_convert.mean():.3f} (mean) | {not_convert.median():.3f} (median)")

# P(convert) distribution
ax = axes[1]
conv_probs    = hold_full[hold_full['y'] == 1]['p_convert']
noconv_probs  = hold_full[hold_full['y'] == 0]['p_convert']

ax.hist(noconv_probs, bins=20, alpha=0.6, color='#d62728', label='Not converted', density=True)
ax.hist(conv_probs,   bins=20, alpha=0.7, color='#2ca02c', label='Converted', density=True)
ax.set_xlabel('Model P(convert) — calibrated')
ax.set_ylabel('Density')
ax.set_title('Predicted P(convert) by Actual Outcome (Holdout)')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../eval/fare_ratio_analysis.png', bbox_inches='tight')
plt.show()
print('Saved → eval/fare_ratio_analysis.png')

## 9. Score Distribution & Threshold Analysis

In [ ]:
thresholds = np.arange(0.02, 0.20, 0.01)
results = []
for t in thresholds:
    preds = (lgbm_cal_hold >= t).astype(int)
    if preds.sum() == 0:
        continue
    tn, fp, fn, tp = confusion_matrix(y_hold, preds).ravel()
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    flagged_pct = preds.mean() * 100
    results.append({'threshold': t, 'precision': precision, 'recall': recall, 'f1': f1, 'flagged_pct': flagged_pct, 'tp': tp, 'fp': fp, 'fn': fn})

thresh_df = pd.DataFrame(results)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(thresh_df['threshold'], thresh_df['precision'], '-o', color='#1f77b4', label='Precision', markersize=4)
ax.plot(thresh_df['threshold'], thresh_df['recall'],    '-s', color='#2ca02c', label='Recall', markersize=4)
ax.plot(thresh_df['threshold'], thresh_df['f1'],        '-^', color='#ff7f0e', label='F1', markersize=4)
ax.set_xlabel('Decision Threshold')
ax.set_ylabel('Score')
ax.set_title('Precision / Recall / F1 vs Threshold (Holdout)')
ax.legend()
ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(thresh_df['threshold'], thresh_df['flagged_pct'], '-o', color='#9467bd', markersize=4)
ax.set_xlabel('Decision Threshold')
ax.set_ylabel('% Requests Flagged as High-Probability')
ax.set_title('Alert Rate vs Threshold (Holdout)')
ax.grid(alpha=0.3)
ax.axhline(100 * float(y_hold.mean()), color='k', linestyle='--', alpha=0.5, label=f'Actual conversion rate ({100*float(y_hold.mean()):.1f}%)')
ax.legend()

plt.tight_layout()
plt.savefig('../eval/threshold_analysis.png', bbox_inches='tight')
plt.show()

best_f1_row = thresh_df.loc[thresh_df['f1'].idxmax()]
print(f"Best F1={best_f1_row['f1']:.3f} at threshold={best_f1_row['threshold']:.2f}")
print(f"  → Precision={best_f1_row['precision']:.3f} | Recall={best_f1_row['recall']:.3f}")
print(f"  → TP={int(best_f1_row['tp'])} | FP={int(best_f1_row['fp'])} | FN={int(best_f1_row['fn'])}")
print('Saved → eval/threshold_analysis.png')

## 10. Lead Time & Conversion Relationship

In [ ]:
all_hist = feat[~feat['is_live'].fillna(False)].dropna(subset=['converted', 'lead_time'])

bins = [0, 7, 14, 30, 60, 90, 180, 365]
labels = ['0-7d', '7-14d', '14-30d', '30-60d', '60-90d', '90-180d', '180d+']
all_hist['lead_bin'] = pd.cut(all_hist['lead_time'], bins=bins, labels=labels)

lead_stats = all_hist.groupby('lead_bin', observed=True).agg(
    total=('converted', 'count'),
    conv_rate=('converted', 'mean')
).reset_index()

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(lead_stats['lead_bin'], lead_stats['conv_rate'] * 100, color='#1f77b4', alpha=0.8)
for bar, total in zip(bars, lead_stats['total']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
            f'n={total}', ha='center', va='bottom', fontsize=9)
ax.set_xlabel('Lead Time (days before departure)')
ax.set_ylabel('Conversion Rate (%)')
ax.set_title('Conversion Rate by Lead Time — All Historical Data')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../eval/lead_time_conversion.png', bbox_inches='tight')
plt.show()
print('Saved → eval/lead_time_conversion.png')

## 11. Agent Conversion Rate Stability

In [ ]:
agent_stats = all_hist.groupby('agent_code').agg(
    requests=('converted', 'count'),
    conv_rate=('converted', 'mean')
).reset_index().sort_values('conv_rate', ascending=False)

# Show top 20 agents by volume
top_agents = agent_stats.nlargest(20, 'requests')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.scatter(agent_stats['requests'], agent_stats['conv_rate'] * 100,
           alpha=0.5, color='#1f77b4', s=40)
ax.set_xlabel('Number of Requests')
ax.set_ylabel('Conversion Rate (%)')
ax.set_title('Agent: Request Volume vs Conversion Rate')
ax.grid(alpha=0.3)

ax = axes[1]
ax.barh(range(len(top_agents)), top_agents['conv_rate'] * 100, color='#1f77b4', alpha=0.8)
ax.set_yticks(range(len(top_agents)))
ax.set_yticklabels([f"{c} (n={r})" for c, r in zip(top_agents['agent_code'], top_agents['requests'])], fontsize=8)
ax.set_xlabel('Conversion Rate (%)')
ax.set_title('Top 20 Agents by Volume — Conversion Rate')
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('../eval/agent_analysis.png', bbox_inches='tight')
plt.show()
print(f'Total agents: {len(agent_stats)}')
print(f'Agents with ≥10 requests: {(agent_stats["requests"] >= 10).sum()}')
print('Saved → eval/agent_analysis.png')

## 12. Summary Dashboard

In [ ]:
fig = plt.figure(figsize=(16, 10))
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# ── ROC ──
ax1 = fig.add_subplot(gs[0, 0])
for y_prob, label, color, ls in [
    (lgbm_cal_hold, 'LGBM cal.', '#1f77b4', '-'),
    (lr_hold,       'LR',        '#ff7f0e', '-.'),
    (base_hold,     'Baseline',  '#d62728', ':'),
]:
    fpr, tpr, _ = roc_curve(y_hold, y_prob)
    auc = roc_auc_score(y_hold, y_prob)
    ax1.plot(fpr, tpr, color=color, linestyle=ls, lw=2, label=f'{label} {auc:.3f}')
ax1.plot([0,1],[0,1],'k--', lw=1, alpha=0.4)
ax1.set_title('ROC Curve'); ax1.legend(fontsize=8); ax1.grid(alpha=0.3)
ax1.set_xlabel('FPR'); ax1.set_ylabel('TPR')

# ── Calibration ──
ax2 = fig.add_subplot(gs[0, 1])
for y_prob, label, color, ls in [
    (lgbm_cal_hold, 'LGBM cal.', '#1f77b4', '-o'),
    (lgbm_raw_hold, 'LGBM raw',  '#aec7e8', '--s'),
    (lr_hold,       'LR',        '#ff7f0e', '-.^'),
]:
    prob_true, prob_pred = calibration_curve(y_hold, y_prob, n_bins=6, strategy='quantile')
    ax2.plot(prob_pred, prob_true, ls, color=color, lw=1.5, markersize=5, label=label)
ax2.plot([0,1],[0,1],'k--', lw=1, alpha=0.4)
ax2.set_title('Calibration'); ax2.legend(fontsize=8); ax2.grid(alpha=0.3)
ax2.set_xlabel('Predicted Prob.'); ax2.set_ylabel('Actual Rate')

# ── SHAP importance ──
ax3 = fig.add_subplot(gs[0, 2])
top8 = importance_df.head(8)
ax3.barh(range(len(top8)), top8['mean_abs_shap'].values[::-1],
         color='#1f77b4', alpha=0.8)
ax3.set_yticks(range(len(top8)))
ax3.set_yticklabels(top8['feature'].values[::-1], fontsize=8)
ax3.set_title('SHAP Importance (Top 8)'); ax3.grid(axis='x', alpha=0.3)
ax3.set_xlabel('Mean |SHAP|')

# ── Fare ratio dist ──
ax4 = fig.add_subplot(gs[1, 0])
if len(converted) > 0:
    ax4.hist(not_convert.clip(0.5, 1.5), bins=15, alpha=0.6, color='#d62728', label='Not conv.', density=True)
    ax4.hist(converted.clip(0.5, 1.5),   bins=15, alpha=0.7, color='#2ca02c', label='Converted', density=True)
ax4.axvline(1.0, color='k', linestyle='--', lw=1, alpha=0.5)
ax4.set_title('Fare Ratio by Outcome'); ax4.legend(fontsize=8); ax4.grid(alpha=0.3)
ax4.set_xlabel('Quoted / Reference Fare')

# ── Score distribution ──
ax5 = fig.add_subplot(gs[1, 1])
ax5.hist(lgbm_cal_hold[y_hold==0], bins=20, alpha=0.6, color='#d62728', label='Not conv.', density=True)
ax5.hist(lgbm_cal_hold[y_hold==1], bins=20, alpha=0.7, color='#2ca02c', label='Converted', density=True)
ax5.set_title('P(convert) by Outcome'); ax5.legend(fontsize=8); ax5.grid(alpha=0.3)
ax5.set_xlabel('Calibrated P(convert)')

# ── Lead time ──
ax6 = fig.add_subplot(gs[1, 2])
ax6.bar(lead_stats['lead_bin'], lead_stats['conv_rate'] * 100, color='#9467bd', alpha=0.8)
ax6.set_title('Conv. Rate by Lead Time'); ax6.grid(axis='y', alpha=0.3)
ax6.set_xlabel('Lead Time'); ax6.set_ylabel('Conv. Rate (%)')
plt.xticks(rotation=30, ha='right', fontsize=8)

fig.suptitle('UC02 Group Booking — Backtest Summary Dashboard\n(November Holdout)', fontsize=14, fontweight='bold')
plt.savefig('../eval/backtest_summary.png', bbox_inches='tight', dpi=130)
plt.show()
print('Saved → eval/backtest_summary.png')

## 13. Final Metrics Table

In [ ]:
print('=' * 65)
print('UC02 GROUP BOOKING ASSISTANT — BACKTEST REPORT')
print('=' * 65)
print(f"Dataset       : {len(feat)} total requests")
print(f"Training      : {len(X_train)} (Jun–Sep) | {int(y_train.sum())} converted")
print(f"Validation    : {len(X_val)} (Oct)       | {int(y_val.sum())} converted")
print(f"Holdout       : {len(X_hold)} (Nov)       | {int(y_hold.sum())} converted")
print(f"Base rate (Nov): {100*float(y_hold.mean()):.1f}%")
print()
print('HOLDOUT PERFORMANCE:')
print(f"  LightGBM (calibrated) ROC-AUC = {roc_auc_score(y_hold, lgbm_cal_hold):.4f}")
print(f"  LightGBM (calibrated) PR-AUC  = {average_precision_score(y_hold, lgbm_cal_hold):.4f}")
print(f"  LightGBM (calibrated) Brier   = {brier_score_loss(y_hold, lgbm_cal_hold):.4f}")
print(f"  Rule-based Baseline   ROC-AUC = {roc_auc_score(y_hold, base_hold):.4f}  (reference)")
print()
print('KEY FINDINGS:')
print('  1. fare_ratio is the dominant predictor — converted requests')
print(f'     avg {converted.mean():.2f}x reference vs {not_convert.mean():.2f}x for non-converted' if len(converted) > 0 else '')
print('  2. Isotonic calibration improves reliability of probability scores')
print('  3. Model outperforms naive baseline on ROC-AUC and PR-AUC')
print('  4. 3% base conversion rate makes this a hard imbalanced classification problem')
print('=' * 65)
print('Generated plots:')
for f in ['roc_pr_curves.png', 'calibration_plot.png', 'shap_importance.png',
          'shap_beeswarm.png', 'fare_ratio_analysis.png', 'threshold_analysis.png',
          'lead_time_conversion.png', 'agent_analysis.png', 'backtest_summary.png']:
    print(f'  eval/{f}')